In [43]:
import sqlite3
import pandas as pd

# Creiamo una connessione a un database in memoria (esiste solo per questa sessione)
# Creiamo UNA SOLA connessione
conn = sqlite3.connect(':memory:')  # NOTA: due punti, non 'memory:'
cursor = conn.cursor()

# DATI GREZZI DI CAMPIONAMENTO GEOLOGICO
data_geologico = {
    'ID_Campione': ['CAMP-001', 'camp-001', 'C002', 'C003 ', 'C004', 'C005', 'C006', 'C007'],
    'Sito': ['Monte Rosa', 'MONTE ROSA', ' Cervino ', 'Cervino', 'Gran Paradiso', 'Monte Bianco', 'CERVINO', 'Monte Rosa'],
    'Data_campionamento': ['2024-06-10', '2024-06-10', '15/06/2024', '2024.06.20', '20-06-2024', '2024/07/01', '01-07-2024', 'ieri'],
    'Litologia': ['Granito', 'GRANITO', ' Scisto ', 'Gneiss', 'Granito', 'Marmo', 'Scisto', 'Granito'],
    'Prof_m': ['2.5', '2.5', '3.0', 'cinque', '1.8', '2.2', 'zero', '3.5'],
    'Au_ppm': ['0.5', '0.5', '1.2', '0.8', 'ND', '2.1', '0.3', '1.5'],
    'Ag_ppm': ['2.1', '2.1', '0.5', '1.1', '0.9', 'ND', '0.2', '0.8'],
    'Note': ['Oro visibile', 'Oro visibile', 'Traccia di Ag', 'Pirite', 'ND', 'Filone quarzoso', 'Scisti mineralizzati', 'Campione duplicato']
}

# Carica in SQLite usando la stessa connessione
df_geo = pd.DataFrame(data_geologico)
df_geo.to_sql('Campioni_Grezzi', conn, index=False, if_exists='replace')  # ← usa conn, non creare nuova

print("=== DATI GREZZI CAMPIONI GEOLOGICI ====")
print(df_geo)
print("\nTabella creata con successo!")

=== DATI GREZZI CAMPIONI GEOLOGICI ====
  ID_Campione           Sito Data_campionamento Litologia  Prof_m Au_ppm  \
0    CAMP-001     Monte Rosa         2024-06-10   Granito     2.5    0.5   
1    camp-001     MONTE ROSA         2024-06-10   GRANITO     2.5    0.5   
2        C002       Cervino          15/06/2024   Scisto      3.0    1.2   
3       C003         Cervino         2024.06.20    Gneiss  cinque    0.8   
4        C004  Gran Paradiso         20-06-2024   Granito     1.8     ND   
5        C005   Monte Bianco         2024/07/01     Marmo     2.2    2.1   
6        C006        CERVINO         01-07-2024    Scisto    zero    0.3   
7        C007     Monte Rosa               ieri   Granito     3.5    1.5   

  Ag_ppm                  Note  
0    2.1          Oro visibile  
1    2.1          Oro visibile  
2    0.5         Traccia di Ag  
3    1.1                Pirite  
4    0.9                    ND  
5     ND       Filone quarzoso  
6    0.2  Scisti mineralizzati  
7    0.8   

## 🎯 Sfide geologiche:
* `Pulizia ID_Campione`: rimuovi spazi e uniforma maiuscole/minuscole
* `Pulizia Sito`: elimina spazi, uniforma uppercase
* `Pulizia Data`: porta tutto a formato YYYY-MM-DD
* `Pulizia Litologia`: trim e uppercase
* `Profondità`: converti in numerico (gestisci 'zero', 'cinque' → NULL o 0?)
* `Au_ppm e Ag_ppm`: converti in numerico (gestisci 'ND' → NULL)

### Analisi EDA:

* Quali siti hanno la media più alta di Au?
* Correlazione tra Au e Ag?
* Quali litologie sono più mineralizzate?
* Distribuzione campioni per sito

In [44]:
# Query di pulizia
q = """
SELECT DISTINCT
 -- Pulizia ID_Campione (-> 'C001')
CASE
WHEN UPPER(TRIM(ID_Campione)) = 'CAMP-001' THEN 'C001'
ELSE UPPER(TRIM(ID_Campione))
END AS id,

 -- Pulizia SITO (-> Cervino)
UPPER(TRIM(Sito)) AS sito,

 -- Pulizia Data_campionamento (-> formato unico YYYY-MM-DD)
CASE
WHEN Data_campionamento LIKE '__/__/____' THEN 
    SUBSTR(Data_campionamento, 7, 4) || '-' ||
    SUBSTR(Data_campionamento, 4, 2) || '-' ||
    SUBSTR(Data_campionamento, 1, 2)
WHEN Data_campionamento LIKE '%.%' THEN 
    REPLACE(Data_campionamento, '.', '-')
WHEN Data_campionamento LIKE '%/%' THEN 
    REPLACE(Data_campionamento, '/', '-')
WHEN Data_campionamento LIKE '__-__-____' THEN 
    SUBSTR(Data_campionamento, 7, 4) || '-' ||
    SUBSTR(Data_campionamento, 4, 2) || '-' ||
    SUBSTR(Data_campionamento, 1, 2)
WHEN Data_campionamento = 'ieri' THEN NULL
ELSE Data_campionamento
END AS data_campionamento,

 -- Pulizia Litologia (-> GRANITO)
UPPER(TRIM(Litologia)) AS litologia,

 -- Pulizia Prof_m (-> 2.5)
CASE
WHEN TRIM(Prof_m) = 'cinque' THEN '5.0'
WHEN TRIM(Prof_m) = 'zero' THEN '0.0'
ELSE Prof_m
END AS profondita_m,

 -- Pulizia Au_ppm (-> 1.5)
CASE
WHEN TRIM(Au_ppm) = 'ND' THEN NULL
ELSE Au_ppm
END AS Au_ppm,

 -- Pulizia Ag_ppm (-> 1.5)
CASE
WHEN TRIM(Ag_ppm) = 'ND' THEN NULL
ELSE Ag_ppm
END AS Ag_ppm,

 -- Pulizia Note (-> 'Abc def')
CASE
WHEN TRIM(Note) = 'ND' THEN NULL
ELSE Note
END AS note

FROM Campioni_Grezzi
"""

# Creiamo la tabella temporanea con i dati puliti (usa cursor dalla cella precedente)
cursor.execute("CREATE TEMPORARY TABLE Campioni_puliti_temp AS " + q)

# Verifichiamo il contenuto
print("\nRilievi Grezzi --> Puliti!")
df_verifica = pd.read_sql_query("SELECT * FROM Campioni_puliti_temp", conn)
print(df_verifica)

# Opzionale: chiudi connessione alla fine
# conn.close()


Rilievi Grezzi --> Puliti!
     id           sito data_campionamento litologia profondita_m Au_ppm  \
0  C001     MONTE ROSA         2024-06-10   GRANITO          2.5    0.5   
1  C002        CERVINO         2024-06-15    SCISTO          3.0    1.2   
2  C003        CERVINO         2024-06-20    GNEISS          5.0    0.8   
3  C004  GRAN PARADISO         2024-06-20   GRANITO          1.8    NaN   
4  C005   MONTE BIANCO         2024-07-01     MARMO          2.2    2.1   
5  C006        CERVINO         2024-07-01    SCISTO          0.0    0.3   
6  C007     MONTE ROSA                NaN   GRANITO          3.5    1.5   

  Ag_ppm                  note  
0    2.1          Oro visibile  
1    0.5         Traccia di Ag  
2    1.1                Pirite  
3    0.9                   NaN  
4    NaN       Filone quarzoso  
5    0.2  Scisti mineralizzati  
6    0.8    Campione duplicato  


In [45]:
print("="*50)
print("Quali siti hanno la media più alta di Au?")
print("="*50)
q_avg_Au = """
SELECT
    sito,
    AVG(Au_ppm) AS media_Au
FROM Campioni_puliti_temp
WHERE Au_ppm IS NOT NULL  -- escludi i NULL
GROUP BY sito
ORDER BY media_Au DESC;    -- dalla più alta alla più bassa
"""
df1 = pd.read_sql_query(q_avg_Au, conn)
print(df1)
print("\n")


print("="*50)
print("Correlazione tra Au e Ag?")
print("="*50)
q_avg_Corr_Au_Ag = """
WITH stats AS (
    SELECT 
        COUNT(*) AS n,
        SUM(Au_ppm) AS sum_x,
        SUM(Ag_ppm) AS sum_y,
        SUM(Au_ppm * Au_ppm) AS sum_x2,
        SUM(Ag_ppm * Ag_ppm) AS sum_y2,
        SUM(Au_ppm * Ag_ppm) AS sum_xy
    FROM Campioni_puliti_temp
    WHERE Au_ppm IS NOT NULL AND Ag_ppm IS NOT NULL
)
SELECT 
    (n * sum_xy - sum_x * sum_y) * 1.0 /
    (SQRT(n * sum_x2 - sum_x * sum_x) *
     SQRT(n * sum_y2 - sum_y * sum_y)) AS correlation_coefficient
FROM stats
"""
df2 = pd.read_sql_query(q_avg_Corr_Au_Ag, conn)
print(df2)
print("\n")


print("="*50)
print("Quali litologie sono più mineralizzate?")
print("="*50)
q_most_mineralization = """
SELECT
    litologia,
    AVG(Au_ppm + Ag_ppm) AS mineralizzazione_media,
    COUNT(*) AS n_campioni
FROM Campioni_puliti_temp
WHERE Au_ppm IS NOT NULL OR Ag_ppm IS NOT NULL
GROUP BY litologia
ORDER BY mineralizzazione_media DESC;
"""
df3 = pd.read_sql_query(q_most_mineralization, conn)
print(df3)
print("\n")


print("="*50)
print("Distribuzione campioni per sito")
print("="*50)
q_distribution = """
SELECT 
sito,
COUNT(*) AS numero_campioni
FROM Campioni_puliti_temp
GROUP BY sito;
"""
df4 = pd.read_sql_query(q_distribution, conn)
print(df4)
print("\n")

Quali siti hanno la media più alta di Au?
           sito  media_Au
0  MONTE BIANCO  2.100000
1    MONTE ROSA  1.000000
2       CERVINO  0.766667


Correlazione tra Au e Ag?
   correlation_coefficient
0                -0.175055


Quali litologie sono più mineralizzate?
  litologia  mineralizzazione_media  n_campioni
0   GRANITO                    2.45           3
1    GNEISS                    1.90           1
2    SCISTO                    1.10           2
3     MARMO                     NaN           1


Distribuzione campioni per sito
            sito  numero_campioni
0        CERVINO                3
1  GRAN PARADISO                1
2   MONTE BIANCO                1
3     MONTE ROSA                2




## 🎯 **Nuove Sfide Geologiche:**

### 1. **Profondità media per sito**
Qual è la profondità media di campionamento per ogni sito? I valori 'zero' e 'cinque' sono stati convertiti correttamente?

### 2. **Campioni ad alto tenore (>1 ppm Au)**
Quali litologie hanno campioni con Au > 1 ppm? Mostra litologia, ID_campione e valore Au.

### 3. **Variabilità dell'Au per sito**
Quale sito ha la maggiore variabilità nell'Au? Calcola deviazione standard per ogni sito.

**💡 Suggerimenti:**
- Per la deviazione standard in SQLite puoi usare la funzione aggregata `STDEV()` oppure calcolarla manualmente
- Ricorda di escludere i valori NULL dove necessario
- Ordina i risultati per rendere chiara la risposta
